# Missingness mechanism tests per state/year file

This notebook reads the nine cleaned csv files in **cleaned_data/cleaned_raw_data** (three states x three election years). For **Age, Total Assets, and Liabilities**, the value **-1** is treated as missing.

The missingness test is run **separately on each of the nine state/year files** (Andhra Pradesh, Bihar, Gujarat x 2014, 2019, 2024). For each file and each target column with missing values, a **Chi-square test of independence** checks whether being missing is associated with each observed candidate characteristic.

In [ ]:
from pathlib import Path
import math
import pandas as pd
import numpy as np

SENTINEL_NUMERIC_COLUMNS = {'Age', 'Total Assets', 'Liabilities'}
MISSINGNESS_TARGET_COLUMNS = ['Age', 'Total Assets', 'Liabilities']
STATES = ['andhra', 'bihar', 'gujarat']
YEARS = [2014, 2019, 2024]
ALPHA = 0.05

def is_missing_value(series, column_name):
    missing = series.isna()
    if column_name in SENTINEL_NUMERIC_COLUMNS:
        numeric_values = pd.to_numeric(series, errors='coerce')
        missing = missing | numeric_values.eq(-1)
    else:
        missing = missing | series.astype(str).str.strip().eq('')
    return missing

CLEANED_DIR = Path('interim/cleaned_raw_data')

input_files = []
for year in YEARS:
    for state in STATES:
        input_files.append(CLEANED_DIR / f'{state}_{year}.csv')

print(f'Number of state/year csv files: {len(input_files)}')
print([path.name for path in input_files])

Number of state/year csv files: 9
['andhra_2014.csv', 'bihar_2014.csv', 'gujarat_2014.csv', 'andhra_2019.csv', 'bihar_2019.csv', 'gujarat_2019.csv', 'andhra_2024.csv', 'bihar_2024.csv', 'gujarat_2024.csv']


## Read the source files

In [ ]:
def read_candidate_file(path):
    frame = pd.read_csv(path)
    return frame


frames_by_file = {}

for input_path in input_files:
    frame = read_candidate_file(input_path)
    frames_by_file[input_path] = frame
    print(f'{input_path.name}: {len(frame)} rows')

print(f'\nColumns: {frame.columns.tolist()}')

andhra_2014.csv: 329 rows
bihar_2014.csv: 605 rows
gujarat_2014.csv: 333 rows
andhra_2019.csv: 314 rows
bihar_2019.csv: 619 rows
gujarat_2019.csv: 371 rows
andhra_2024.csv: 450 rows
bihar_2024.csv: 496 rows
gujarat_2024.csv: 266 rows

Columns: ['Sno.', 'Candidate', 'Party', 'Criminal Cases', 'Education', 'Age', 'Total Assets', 'Liabilities']


## Missingness by state and year

In [ ]:
missingness_rows = []

for input_path, frame in frames_by_file.items():
    file_state, file_year_text = input_path.stem.rsplit('_', 1)
    row_count = len(frame)

    for column in frame.columns:
        missing_count = int(is_missing_value(frame[column], column).sum())

        if row_count == 0:
            missing_percent = 0.0
        else:
            missing_percent = (missing_count / row_count) * 100

        missingness_rows.append(
            {
                'Source File': input_path.name,
                'State': file_state,
                'Year': int(file_year_text),
                'Column': column,
                'Row Count': row_count,
                'Missing Count': missing_count,
                'Missing Percent': round(missing_percent, 2),
            }
        )

missingness_report = pd.DataFrame(missingness_rows)

target_missingness = missingness_report[missingness_report['Column'].isin(MISSINGNESS_TARGET_COLUMNS)]
missing_by_year_state = target_missingness.pivot_table(
    index=['Year', 'State'],
    columns='Column',
    values='Missing Count',
    aggfunc='sum',
)[MISSINGNESS_TARGET_COLUMNS]
missing_by_year_state['Row Count'] = target_missingness.groupby(['Year', 'State'])['Row Count'].first()

missing_by_year = missing_by_year_state.groupby(level='Year').sum()
for column in MISSINGNESS_TARGET_COLUMNS:
    missing_by_year[f'{column} %'] = (missing_by_year[column] / missing_by_year['Row Count'] * 100).round(2)

display(missing_by_year_state)
display(missing_by_year)

Column        Age  Total Assets  Liabilities  Row Count
Year State                                             
2014 andhra     7             8            0        329
     bihar      6             2            0        605
     gujarat    5             1            0        333
2019 andhra     3             5            0        314
     bihar      0             2            0        619
     gujarat    0             1            0        371
2024 andhra     0            11            0        450
     bihar      0             0            0        496
     gujarat    0             0            0        266

Column,Age,Total Assets,Liabilities,Row Count,Age %,Total Assets %,Liabilities %
Year,,,,,,,
2014,18,11,0,1267,1.42,0.87,0
2019,3,8,0,1304,0.23,0.61,0
2024,0,11,0,1212,0,0.91,0


## Candidate-level analysis frame

All nine files are stacked into one frame for convenience. Every test below filters this frame to a single state/year file, so each file is tested on its own. 
The observed features are:

- Independent: party is IND
- Education Level: ordinal 0 (Illiterate) to 9 (Doctorate); Others / Not Given are left missing
- Has Criminal Case: at least one declared criminal case
- Log Liabilities and Zero Liabilities: liabilities are never missing, so they are a fully observed financial proxy
- Age and Log Total Assets: the *other* target, used as a covariate only where it is observed

In [29]:
EDUCATION_LEVELS = {
    'Illiterate': 0,
    'Literate': 1,
    '5th Pass': 2,
    '8th Pass': 3,
    '10th Pass': 4,
    '12th Pass': 5,
    'Graduate': 6,
    'Graduate Professional': 7,
    'Post Graduate': 8,
    'Doctorate': 9,
}

candidate_parts = []

for input_path, source_frame in frames_by_file.items():
    file_state, file_year_text = input_path.stem.rsplit('_', 1)
    part = pd.DataFrame(index=source_frame.index)
    part['State'] = file_state
    part['Year'] = int(file_year_text)
    part['Independent'] = source_frame['Party'].astype(str).str.strip().eq('IND').astype(float)
    part['Education Level'] = source_frame['Education'].astype(str).str.strip().map(EDUCATION_LEVELS)
    part['Has Criminal Case'] = pd.to_numeric(source_frame['Criminal Cases'], errors='coerce').gt(0).astype(float)

    age_values = pd.to_numeric(source_frame['Age'], errors='coerce')
    asset_values = pd.to_numeric(source_frame['Total Assets'], errors='coerce')
    liability_values = pd.to_numeric(source_frame['Liabilities'], errors='coerce')

    part['Age'] = age_values.mask(age_values.eq(-1))
    part['Log Total Assets'] = np.log1p(asset_values.mask(asset_values.lt(0)))
    part['Log Liabilities'] = np.log1p(liability_values.mask(liability_values.lt(0)))
    part['Zero Liabilities'] = liability_values.eq(0).astype(float)

    for column in MISSINGNESS_TARGET_COLUMNS:
        part[f'{column} Missing'] = is_missing_value(source_frame[column], column).astype(int)

    candidate_parts.append(part)

candidates = pd.concat(candidate_parts, ignore_index=True)
print(f'Candidates: {len(candidates)}')
display(candidates.head())

Candidates: 3783


,State,Year,Independent,Education Level,Has Criminal Case,Age,Log Total Assets,Log Liabilities,Zero Liabilities,Age Missing,Total Assets Missing,Liabilities Missing
0,andhra,2014,0,4,0,49,16.68,15.15,0,0,0,0
1,andhra,2014,0,4,0,44,15.85,15.76,0,0,0,0
2,andhra,2014,0,5,0,64,20.97,18.7,0,0,0,0
3,andhra,2014,0,8,1,63,16.35,14.73,0,0,0,0
4,andhra,2014,0,4,0,33,13.38,0,1,0,0,0


## Missingness patterns by year
Which target fields are missing together, per year and state.

In [ ]:
def describe_missing_pattern(row):
    missing_fields = [
        column
        for column in MISSINGNESS_TARGET_COLUMNS
        if row[f'{column} Missing'] == 1
    ]
    return ' + '.join(missing_fields) if missing_fields else 'No target values missing'


candidates['Missing Pattern'] = candidates.apply(describe_missing_pattern, axis=1)

missingness_patterns = (
    candidates.groupby(['Year', 'State', 'Missing Pattern'])
    .size()
    .rename('Candidate Count')
    .reset_index()
)

display(missingness_patterns.pivot_table(
    index=['Year', 'State'],
    columns='Missing Pattern',
    values='Candidate Count',
    fill_value=0,
))

Missing Pattern  Age  Age + Total Assets  No target values missing  \
Year State                                                           
2014 andhra        6                   1                       315   
     bihar         6                   0                       597   
     gujarat       5                   0                       327   
2019 andhra        3                   0                       306   
     bihar         0                   0                       617   
     gujarat       0                   0                       370   
2024 andhra        0                   0                       439   
     bihar         0                   0                       496   
     gujarat       0                   0                       266   

Missing Pattern  Total Assets  
Year State                     
2014 andhra                 7  
     bihar                  2  
     gujarat                1  
2019 andhra                 5  
     bihar                  2  
     gujarat                1  
2024 andhra                11  
     bihar                  0  
     gujarat                0

## Chi-square helpers
Helper functions to find the p-value for a chi-square test
Chi-square test checks for independence across the features

In [31]:
import numpy as np
from scipy.stats import chi2_contingency

def chi_square(table):
    observed = table.to_numpy(dtype=float)
    observed = observed[:, observed.sum(axis=0) > 0]
    
    # Run SciPy's chi-square test of independence (correction=False matches uncorrected Pearson's test)
    statistic, pvalue, degrees_of_freedom, expected = chi2_contingency(observed, correction=False)
    
    return float(statistic), int(degrees_of_freedom), float(pvalue), float(expected.min())

## Chi-square tests of independence, per state/year file

| Feature | Categories |
|---|---|
| Education | Below 10th / 10th-12th / Graduate or higher / Others or not given |
| Criminal Case | Yes / No |
| Liabilities | Zero / Positive |
| Age Group (for asset missingness) | Under 35 / 35-50 / Over 50, among candidates with a declared age |
| Asset Tercile (for age missingness) | Low / middle / high within the file, among candidates with declared assets |

H0: missingness is independent of the feature. 
Rejection (p < 0.05) means missingness depends on observed information, which rules out MCAR and is consistent with MAR. It cannot rule out MNAR. 
**Missing Rate by Category** shows the direction of any association. Because missing values are rare, the expected count in the missing row is often below 5; 
**Min Expected Count** flags these cases, where the p-value is only approximate.

## Mechanism classification per state/year file

Decision rule for each file and column:

1. **Not applicable** if nothing is missing.
2. **MAR** if any chi-square test rejects independence at 5% **and** that test is usable: A test is deemed usable when the no of missing values is large enough to compare thus when the expected count is below 1 its listed unreliable and is not counted. Missingness depends on observed variables. This cannot formally exclude MNAR.
3. **Consistent with MCAR** otherwise. With only a handful of missing rows, this means "no evidence against MCAR", not proof of MCAR. Files with fewer than 5 missing values are flagged as having very low power.

MNAR cannot be tested from observed data; it is discussed in the conclusions as a plausibility argument.

In [ ]:
LOW_POWER_THRESHOLD = 5
MIN_EXPECTED_FOR_VALID_TEST = 1.0
summary_rows = []

for year, state in FILE_KEYS:
    file_data = file_frame(year, state)

    for column in MISSINGNESS_TARGET_COLUMNS:
        missing_count = int(file_data[f'{column} Missing'].sum())
        row = {
            'Year': year,
            'State': state,
            'Column': column,
            'Missing Count': missing_count,
            'Missing Percent': round(missing_count / len(file_data) * 100, 2),
        }

        if missing_count == 0:
            summary_rows.append({**row, 'Mechanism': 'Not applicable', 'Evidence': 'No missing values'})
            continue

        column_tests = chi_square_report[
            (chi_square_report['Year'] == year)
            & (chi_square_report['State'] == state)
            & (chi_square_report['Column'] == column)
        ]
        rejected = column_tests[column_tests['p-value'] < ALPHA]
        significant = rejected[rejected['Min Expected Count'] >= MIN_EXPECTED_FOR_VALID_TEST]
        unreliable = rejected[rejected['Min Expected Count'] < MIN_EXPECTED_FOR_VALID_TEST]
        unreliable_text = (
            ' Unreliable rejection (expected count < 1): ' + ', '.join(unreliable['Feature']) + '.'
            if not unreliable.empty
            else ''
        )
        p_value_text = ', '.join(
            f'{feature} p={p_value:.3g}'
            for feature, p_value in zip(column_tests['Feature'], column_tests['p-value'])
        )

        if not significant.empty:
            mechanism = 'MAR (MNAR not excluded)'
            evidence = 'Independence rejected for ' + ', '.join(significant['Feature']) + f'.{unreliable_text} {p_value_text}'
        elif missing_count < LOW_POWER_THRESHOLD:
            mechanism = 'Consistent with MCAR (very low power)'
            evidence = f'No valid test rejected, but only {missing_count} missing.{unreliable_text} {p_value_text}'
        else:
            mechanism = 'Consistent with MCAR'
            evidence = f'No valid test rejected.{unreliable_text} {p_value_text}'

        summary_rows.append({**row, 'Mechanism': mechanism, 'Evidence': evidence})

mechanism_summary = pd.DataFrame(summary_rows)

with pd.option_context('display.max_colwidth', None):
    display(mechanism_summary[mechanism_summary['Missing Count'] > 0])

display(mechanism_summary.pivot_table(
    index=['Year', 'State'],
    columns='Column',
    values='Mechanism',
    aggfunc='first',
)[MISSINGNESS_TARGET_COLUMNS])

,Year,State,Column,Missing Count,Missing Percent,Mechanism,Evidence
0,2014,andhra,Age,7,2.13,Consistent with MCAR,"No valid test rejected. Education p=0.825, Criminal Case p=0.427, Liabilities p=0.82, Asset Tercile p=0.601"
1,2014,andhra,Total Assets,8,2.43,MAR (MNAR not excluded),"Independence rejected for Liabilities. Education p=0.168, Criminal Case p=0.19, Liabilities p=0.00241, Age Group p=0.912"
3,2014,bihar,Age,6,0.99,Consistent with MCAR,"No valid test rejected. Education p=0.9, Criminal Case p=0.897, Liabilities p=0.592, Asset Tercile p=0.603"
4,2014,bihar,Total Assets,2,0.33,Consistent with MCAR (very low power),"No valid test rejected, but only 2 missing. Education p=0.549, Criminal Case p=0.558, Liabilities p=0.208, Age Group p=0.294"
6,2014,gujarat,Age,5,1.5,Consistent with MCAR,"No valid test rejected. Education p=0.432, Criminal Case p=0.263, Liabilities p=0.0645, Asset Tercile p=0.242"
7,2014,gujarat,Total Assets,1,0.3,Consistent with MCAR (very low power),"No valid test rejected, but only 1 missing. Education p=0.609, Criminal Case p=0.619, Liabilities p=0.411, Age Group p=0.501"
9,2019,andhra,Age,3,0.96,Consistent with MCAR (very low power),"No valid test rejected, but only 3 missing. Education p=0.441, Criminal Case p=0.369, Liabilities p=0.792, Asset Tercile p=0.364"
10,2019,andhra,Total Assets,5,1.59,MAR (MNAR not excluded),"Independence rejected for Liabilities. Unreliable rejection (expected count < 1): Age Group. Education p=0.734, Criminal Case p=0.955, Liabilities p=0.00658, Age Group p=0.0289"
13,2019,bihar,Total Assets,2,0.32,Consistent with MCAR (very low power),"No valid test rejected, but only 2 missing. Unreliable rejection (expected count < 1): Education. Education p=1.33e-06, Criminal Case p=0.348, Liabilities p=0.207, Age Group p=0.468"
16,2019,gujarat,Total Assets,1,0.27,Consistent with MCAR (very low power),"No valid test rejected, but only 1 missing. Education p=0.509, Criminal Case p=0.666, Liabilities p=0.399, Age Group p=0.45"


Column                                          Age  \
Year State                                            
2014 andhra                    Consistent with MCAR   
     bihar                     Consistent with MCAR   
     gujarat                   Consistent with MCAR   
2019 andhra   Consistent with MCAR (very low power)   
     bihar                           Not applicable   
     gujarat                         Not applicable   
2024 andhra                          Not applicable   
     bihar                           Not applicable   
     gujarat                         Not applicable   

Column                                 Total Assets     Liabilities  
Year State                                                           
2014 andhra                 MAR (MNAR not excluded)  Not applicable  
     bihar    Consistent with MCAR (very low power)  Not applicable  
     gujarat  Consistent with MCAR (very low power)  Not applicable  
2019 andhra                 MAR (MNAR not excluded)  Not applicable  
     bihar    Consistent with MCAR (very low power)  Not applicable  
     gujarat  Consistent with MCAR (very low power)  Not applicable  
2024 andhra                 MAR (MNAR not excluded)  Not applicable  
     bihar                           Not applicable  Not applicable  
     gujarat                         Not applicable  Not applicable

## Conclusions:

Each of the nine files was tested on its own with chi-square tests of independence between the missingness indicator and each observed characteristic.  Liabilities is never missing in any file.
### Total Assets: MAR where testable, with MNAR plausible

The three Andhra files hold 24 of the 30 missing asset values, and they are the only ones where the chi-square test has enough cases to detect anything:

- **Andhra 2014:** all 8 missing values are among candidates with zero liabilities.
- **Andhra 2019:** all 5 are among zero-liability candidates.
  - The age-group test also gives p = 0.03. Its smallest expected count is below 1, so it is treated as unreliable and not counted.
- **Andhra 2024:** independents are missing assets at 4.8% vs 0.8% for party candidates (p = 0.006).

The other four files with missing assets (Bihar 2014, Bihar 2019, Gujarat 2014, Gujarat 2019) have only 1-2 missing values each, so no valid chi-square test can reject. Every one of those 6 candidates also has zero liabilities, the same profile as in Andhra. The most likely reading is the same mechanism, too rare to detect.

- **Bihar 2019** The test showed a high significant result (p=1e-6). It happened simply because there was only 1 missing piece of data in a tiny group of 10 candidates, which tricked the math formula into showing a fake pattern.

Within each testable file, missing assets depend on observed variables (no liabilities, independent status), so the mechanism being **MAR** is plausible.

**MNAR is still plausible.** If candidates with very small assets are more likely to leave the declaration blank, missingness also depends on the asset value itself beyond what liabilities and party explain. A chi-square test on observed data cannot tell this apart from MAR.

**Implications:**

Considering all the above evidences, we decided to drop the missing rows due to the low count as the tests couldnt be deemed reliable

### Age: mostly MCAR

- **Andhra 2014 and Bihar 2014:** 7 and 6 missing, and no chi-square test is significant (all p > 0.2). Missing-age candidates are spread evenly across party type, education, criminal record, liabilities, serial position and asset level.
- **Andhra 2019:** only 3 missing and nothing rejects. MCAR-consistent, with very little power (no of missing fields).
- **Gujarat 2014: MAR on party type.** All 5 missing ages are independents (3.2% of independents vs 0 of 175 party candidates, p = 0.018).
  - Liabilities (p = 0.06) is borderline.
  - With 66 chi-square tests run at the 5% level, a few rejections are expected by chance, so this single moderate result is weak evidence.

Age is not something candidates would hide, and missing-age candidates are not concentrated in any asset tercile in any file, so MNAR is unlikely.

### Caveats

- **Small expected counts.** Missing counts per file are tiny (1-11), so the expected count in the "missing" row is below 5 in almost every table and the chi-square p-values are approximate. Rejections with an expected count below 1 were discarded. The rejections that are kept (Andhra 2014, 2019 and 2024 assets; Gujarat 2014 age) all have minimum expected counts of about 2-5.
- **MAR vs MNAR.** No test on observed data can separate MAR from MNAR. The MNAR discussion above is a plausibility argument.